In [ ]:
from FIT_python.plot_style import apply_style
apply_style()
import pandas as pd
from FIT_python.pipeline_individual_id.utils import sequential_holdout_ids
from FIT_python.pipeline_individual_id.generate_trails_and_trailpairs import generate_pairwise_comparisons_from_df
from FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaseline
from FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umap
from FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamese
from FIT_python.pipeline_individual_id.evaluation import compute_confusion
import seaborn as sns, matplotlib.pyplot as plt
df = pd.read_parquet('data/cleaned/Eurasian_Otter.parquet')
feature_cols = [c for c in df.columns if c.startswith(('dist','ang','t','v')) and c!='trail']
ids = df['individual_id'].unique()
split = sequential_holdout_ids(ids, val_sizes=[3,6,9], n_iter=1, random_state=0)[0]
train_df = df[df['individual_id'].isin(split['train_ids'])]
val_df = df[df['individual_id'].isin(split['val_ids'])]
val_comps, _ = generate_pairwise_comparisons_from_df(val_df)


In [ ]:
res_base = DistanceBaseline.run(train_df, val_comps, feature_cols)
Xb = res_base.filter(regex='^dist_')
yb = res_base['same_individual'].map({'True':1,'False':0})
from sklearn.linear_model import LogisticRegression
clf_base = LogisticRegression(max_iter=200).fit(Xb, yb)
pred_base = clf_base.predict(Xb)
cm_base = compute_confusion(pd.DataFrame({'same_individual':yb,'pred':pred_base}))
sns.heatmap(cm_base, annot=True, fmt='d'); plt.show()


In [ ]:
res_umap, emb_umap = run_umap(train_df, val_comps, feature_cols)
yu = res_umap['same_individual'].map({'True':1,'False':0})
pred_u = (res_umap['pred_same_proba']>0.5).astype(int)
cm_umap = compute_confusion(pd.DataFrame({'same_individual':yu,'pred':pred_u}))
sns.heatmap(cm_umap, annot=True, fmt='d'); plt.show()


In [ ]:
res_siam = run_siamese(train_df, val_comps, feature_cols)
ys = pd.Series([c['same_individual'] for c in val_comps]).map({'True':1,'False':0})
pred_s = (pd.Series([r['pred_same'] for r in res_siam])>0.5).astype(int)
cm_siam = compute_confusion(pd.DataFrame({'same_individual':ys,'pred':pred_s}))
sns.heatmap(cm_siam, annot=True, fmt='d'); plt.show()
